In [3]:
# Installation (uniquement si pas déjà présent dans Colab)
!pip install xgboost prophet --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print("✅ Toutes les bibliothèques sont chargées")

✅ Toutes les bibliothèques sont chargées


In [4]:
df = pd.read_csv('final_salesdaily_cleaned.csv')

print("Shape:", df.shape)
print("\nColonnes:", df.columns.tolist())
print("\nAperçu:")
df.head()


Shape: (2106, 29)

Colonnes: ['datum', 'M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06', 'Year', 'Month', 'Weekday Name', 'M01AB_stock', 'M01AB_stock_status', 'M01AE_stock', 'M01AE_stock_status', 'N02BA_stock', 'N02BA_stock_status', 'N02BE_stock', 'N02BE_stock_status', 'N05B_stock', 'N05B_stock_status', 'N05C_stock', 'N05C_stock_status', 'R03_stock', 'R03_stock_status', 'R06_stock', 'R06_stock_status', 'Day']

Aperçu:


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,Year,...,N02BE_stock_status,N05B_stock,N05B_stock_status,N05C_stock,N05C_stock_status,R03_stock,R03_stock_status,R06_stock,R06_stock_status,Day
0,2014-01-02,0,3,3,32,7,0,0,2,2014,...,High Stock,357,High Stock,449,High Stock,350,High Stock,498,High Stock,2
1,2014-01-03,8,4,4,50,16,0,20,4,2014,...,High Stock,341,High Stock,449,High Stock,330,High Stock,494,High Stock,3
2,2014-01-04,2,1,6,61,10,0,9,1,2014,...,High Stock,331,High Stock,449,High Stock,321,High Stock,493,High Stock,4
3,2014-01-05,4,3,7,41,8,0,3,0,2014,...,High Stock,323,High Stock,449,High Stock,318,High Stock,493,High Stock,5
4,2014-01-06,5,1,4,21,16,2,6,2,2014,...,High Stock,307,High Stock,447,High Stock,312,High Stock,491,High Stock,6


In [5]:
print("=== Valeurs manquantes ===")
print(df.isnull().sum())

=== Valeurs manquantes ===
datum                 0
M01AB                 0
M01AE                 0
N02BA                 0
N02BE                 0
N05B                  0
N05C                  0
R03                   0
R06                   0
Year                  0
Month                 0
Weekday Name          0
M01AB_stock           0
M01AB_stock_status    0
M01AE_stock           0
M01AE_stock_status    0
N02BA_stock           0
N02BA_stock_status    0
N02BE_stock           0
N02BE_stock_status    0
N05B_stock            0
N05B_stock_status     0
N05C_stock            0
N05C_stock_status     0
R03_stock             0
R03_stock_status      0
R06_stock             0
R06_stock_status      0
Day                   0
dtype: int64


In [6]:
print("\n=== Types des colonnes ===")
print(df.dtypes)


=== Types des colonnes ===
datum                 object
M01AB                  int64
M01AE                  int64
N02BA                  int64
N02BE                  int64
N05B                   int64
N05C                   int64
R03                    int64
R06                    int64
Year                   int64
Month                  int64
Weekday Name          object
M01AB_stock            int64
M01AB_stock_status    object
M01AE_stock            int64
M01AE_stock_status    object
N02BA_stock            int64
N02BA_stock_status    object
N02BE_stock            int64
N02BE_stock_status    object
N05B_stock             int64
N05B_stock_status     object
N05C_stock             int64
N05C_stock_status     object
R03_stock              int64
R03_stock_status      object
R06_stock              int64
R06_stock_status      object
Day                    int64
dtype: object


In [7]:
# Convertir la date en datetime
df['datum'] = pd.to_datetime(df['datum'])

In [8]:
print("\n=== Période couverte ===")
print(f"De {df['datum'].min()} à {df['datum'].max()}")
print(f"\n✅ Données prêtes")


=== Période couverte ===
De 2014-01-02 00:00:00 à 2019-10-08 00:00:00

✅ Données prêtes


In [9]:
df['datum'] = pd.to_datetime(df['datum'])
df = df.sort_values('datum').reset_index(drop=True)

meds = ['M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06']

# ── Features temporelles cycliques ──────────────────────────────────────────
df['day_of_week']    = df['datum'].dt.dayofweek
df['month']          = df['datum'].dt.month
df['week_of_year']   = df['datum'].dt.isocalendar().week.astype(int)
df['quarter']        = df['datum'].dt.quarter
df['is_weekend']     = (df['datum'].dt.dayofweek >= 5).astype(int)
df['is_month_start'] = df['datum'].dt.is_month_start.astype(int)
df['is_month_end']   = df['datum'].dt.is_month_end.astype(int)

# Encodage cyclique (lundi et dimanche sont adjacents dans le cycle)
df['dow_sin']    = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos']    = np.cos(2 * np.pi * df['day_of_week'] / 7)
df['month_sin']  = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos']  = np.cos(2 * np.pi * df['month'] / 12)
df['week_sin']   = np.sin(2 * np.pi * df['week_of_year'] / 52)
df['week_cos']   = np.cos(2 * np.pi * df['week_of_year'] / 52)

# ── Lag + Rolling (inchangé) ─────────────────────────────────────────────────
for med in meds:
    df[f'{med}_lag1']   = df[med].shift(1)
    df[f'{med}_lag7']   = df[med].shift(7)
    df[f'{med}_lag14']  = df[med].shift(14)
    df[f'{med}_lag30']  = df[med].shift(30)
    df[f'{med}_roll7']  = df[med].shift(1).rolling(7).mean()
    df[f'{med}_roll14'] = df[med].shift(1).rolling(14).mean()
    df[f'{med}_roll30'] = df[med].shift(1).rolling(30).mean()
    df[f'{med}_std7']   = df[med].shift(1).rolling(7).std()
    # Différenciation — aide les séries très bruitées
    df[f'{med}_diff1']  = df[med].diff(1)
    df[f'{med}_diff7']  = df[med].diff(7)

df = df.dropna().reset_index(drop=True)
print(f"Données après feature engineering : {df.shape}")

Données après feature engineering : (2076, 122)


In [10]:
# Nouvelles base_features (remplace l'ancien LabelEncoder)
base_features = [
    'dow_sin', 'dow_cos',
    'month_sin', 'month_cos',
    'week_sin', 'week_cos',
    'quarter', 'is_weekend',
    'is_month_start', 'is_month_end'
]

split_index = int(len(df) * 0.80)
train_df = df.iloc[:split_index]
test_df  = df.iloc[split_index:]
print(f"Train : {len(train_df)} jours | Test : {len(test_df)} jours")

Train : 1660 jours | Test : 416 jours


In [11]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import xgboost as xgb

def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return round(np.mean(np.abs((y_true[mask]-y_pred[mask])/y_true[mask]))*100, 2)

results = {}

for med in meds:
    med_features = base_features + [
        f'{med}_lag1',  f'{med}_lag7',  f'{med}_lag14', f'{med}_lag30',
        f'{med}_roll7', f'{med}_roll14', f'{med}_roll30', f'{med}_std7',
        f'{med}_diff1', f'{med}_diff7'
    ]

    X_train, X_test = train_df[med_features], test_df[med_features]
    y_train, y_test = train_df[med],          test_df[med]

    rf = RandomForestRegressor(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=5,
        max_features=0.7,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_test)

    val_size = int(len(X_train) * 0.2)
    X_tr, X_val = X_train.iloc[:-val_size], X_train.iloc[-val_size:]
    y_tr, y_val = y_train.iloc[:-val_size], y_train.iloc[-val_size:]

    xgb_model = xgb.XGBRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        early_stopping_rounds=30,
        random_state=42,
        verbosity=0
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    y_pred_xgb = xgb_model.predict(X_test)

    r2_rf, r2_xgb = r2_score(y_test, y_pred_rf), r2_score(y_test, y_pred_xgb)
    mape_rf, mape_xgb = mape(y_test, y_pred_rf), mape(y_test, y_pred_xgb)
    best = 'XGBoost' if r2_xgb > r2_rf else 'RF'
    best_pred = y_pred_xgb if r2_xgb > r2_rf else y_pred_rf

    results[med] = {
        'rf_mae':   round(mean_absolute_error(y_test, y_pred_rf), 2),
        'rf_mape':  mape_rf,
        'rf_r2':    round(r2_rf, 3),
        'xgb_mae':  round(mean_absolute_error(y_test, y_pred_xgb), 2),
        'xgb_mape': mape_xgb,
        'xgb_r2':   round(r2_xgb, 3),
        'best': best, 'best_pred': best_pred, 'y_test': y_test.values,
        'rf_model': rf, 'xgb_model': xgb_model, 'med_features': med_features
    }
    print(f"{med:6s} | RF R²: {r2_rf:+.3f} MAPE: {mape_rf}% | XGB R²: {r2_xgb:+.3f} MAPE: {mape_xgb}% | Meilleur: {best}")

M01AB  | RF R²: +0.963 MAPE: 6.37% | XGB R²: +0.987 MAPE: 3.02% | Meilleur: XGBoost
M01AE  | RF R²: +0.943 MAPE: 5.94% | XGB R²: +0.981 MAPE: 3.06% | Meilleur: XGBoost
N02BA  | RF R²: +0.979 MAPE: 5.81% | XGB R²: +0.994 MAPE: 3.55% | Meilleur: XGBoost
N02BE  | RF R²: +0.949 MAPE: 7.2% | XGB R²: +0.983 MAPE: 4.11% | Meilleur: XGBoost
N05B   | RF R²: +0.946 MAPE: 10.66% | XGB R²: +0.989 MAPE: 4.04% | Meilleur: XGBoost
N05C   | RF R²: +0.983 MAPE: 5.8% | XGB R²: +0.997 MAPE: 2.32% | Meilleur: XGBoost
R03    | RF R²: +0.938 MAPE: 18.02% | XGB R²: +0.956 MAPE: 11.38% | Meilleur: XGBoost
R06    | RF R²: +0.927 MAPE: 12.25% | XGB R²: +0.977 MAPE: 5.12% | Meilleur: XGBoost


In [12]:
rows = []
for med, m in results.items():
    rows.append({
        'Médicament' : med,
        'RF MAE'     : m['rf_mae'],
        'RF MAPE(%)'  : m['rf_mape'],
        'RF R²'      : m['rf_r2'],
        'XGB MAE'    : m['xgb_mae'],
        'XGB MAPE(%)' : m['xgb_mape'],
        'XGB R²'     : m['xgb_r2'],
        'Meilleur'   : m['best']
    })

metrics_df = pd.DataFrame(rows)
print(metrics_df.to_string(index=False))

Médicament  RF MAE  RF MAPE(%)  RF R²  XGB MAE  XGB MAPE(%)  XGB R² Meilleur
     M01AB    0.29        6.37  0.963     0.15         3.02   0.987  XGBoost
     M01AE    0.23        5.94  0.943     0.12         3.06   0.981  XGBoost
     N02BA    0.17        5.81  0.979     0.10         3.55   0.994  XGBoost
     N02BE    2.09        7.20  0.949     1.22         4.11   0.983  XGBoost
      N05B    0.63       10.66  0.946     0.30         4.04   0.989  XGBoost
      N05C    0.05        5.80  0.983     0.02         2.32   0.997  XGBoost
       R03    1.17       18.02  0.938     0.86        11.38   0.956  XGBoost
       R06    0.41       12.25  0.927     0.19         5.12   0.977  XGBoost


In [13]:
# Supprimer l'ancien LabelEncoder et reconstruire les features temporelles
df['day_of_week']    = df['datum'].dt.dayofweek
df['month']          = df['datum'].dt.month
df['week_of_year']   = df['datum'].dt.isocalendar().week.astype(int)
df['quarter']        = df['datum'].dt.quarter
df['is_weekend']     = (df['datum'].dt.dayofweek >= 5).astype(int)
df['is_month_start'] = df['datum'].dt.is_month_start.astype(int)

# Encodage cyclique — c'est la clé
df['dow_sin']   = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos']   = np.cos(2 * np.pi * df['day_of_week'] / 7)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

base_features = [
    'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'week_of_year', 'quarter', 'is_weekend', 'is_month_start'
]

In [14]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=6,           # limiter la profondeur
    min_samples_leaf=5,    # forcer la généralisation
    max_features=0.7,      # sous-échantillonnage des features
    random_state=42,
    n_jobs=-1
)

In [15]:
# Modéliser la variation journalière plutôt que la valeur absolue
for med in meds:
    df[f'{med}_diff1'] = df[med].diff(1)   # variation vs hier
    df[f'{med}_diff7'] = df[med].diff(7)   # variation vs même jour semaine passée

# Dans la boucle d'entraînement, prédire la différence puis reconstituer
# y_train = train_df[med].diff(1).dropna()
# prediction finale = last_known_value + predicted_diff

In [16]:
# Diagnostiquer la taille effective après feature engineering
print(f"Lignes utilisables : {len(df)}")
print(f"Train : {split_index} | Test : {len(df) - split_index}")

# Si test < 60 jours, le R² est peu fiable — vérifier la période totale
print(f"Période test : {test_df['datum'].min().date()} → {test_df['datum'].max().date()}")

Lignes utilisables : 2076
Train : 1660 | Test : 416
Période test : 2018-08-19 → 2019-10-08


In [17]:
print(df.columns.tolist())
print(df[['M01AB_stock', 'M01AE_stock', 'N02BA_stock',
          'N02BE_stock', 'N05B_stock', 'N05C_stock',
          'R03_stock', 'R06_stock']].tail(3))

['datum', 'M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06', 'Year', 'Month', 'Weekday Name', 'M01AB_stock', 'M01AB_stock_status', 'M01AE_stock', 'M01AE_stock_status', 'N02BA_stock', 'N02BA_stock_status', 'N02BE_stock', 'N02BE_stock_status', 'N05B_stock', 'N05B_stock_status', 'N05C_stock', 'N05C_stock_status', 'R03_stock', 'R03_stock_status', 'R06_stock', 'R06_stock_status', 'Day', 'day_of_week', 'month', 'week_of_year', 'quarter', 'is_weekend', 'is_month_start', 'is_month_end', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'week_sin', 'week_cos', 'M01AB_lag1', 'M01AB_lag7', 'M01AB_lag14', 'M01AB_lag30', 'M01AB_roll7', 'M01AB_roll14', 'M01AB_roll30', 'M01AB_std7', 'M01AB_diff1', 'M01AB_diff7', 'M01AE_lag1', 'M01AE_lag7', 'M01AE_lag14', 'M01AE_lag30', 'M01AE_roll7', 'M01AE_roll14', 'M01AE_roll30', 'M01AE_std7', 'M01AE_diff1', 'M01AE_diff7', 'N02BA_lag1', 'N02BA_lag7', 'N02BA_lag14', 'N02BA_lag30', 'N02BA_roll7', 'N02BA_roll14', 'N02BA_roll30', 'N02BA_std7', 'N02BA_diff1

In [18]:
stock_cols = {
    'M01AB': 'M01AB_stock', 'M01AE': 'M01AE_stock',
    'N02BA': 'N02BA_stock', 'N02BE': 'N02BE_stock',
    'N05B' : 'N05B_stock',  'N05C' : 'N05C_stock',
    'R03'  : 'R03_stock',   'R06'  : 'R06_stock'
}

current_stock = {med: df[col].iloc[-1] for med, col in stock_cols.items()}
print(current_stock)

{'M01AB': np.int64(230), 'M01AE': np.int64(133), 'N02BA': np.int64(345), 'N02BE': np.int64(104), 'N05B': np.int64(125), 'N05C': np.int64(177), 'R03': np.int64(203), 'R06': np.int64(109)}


In [19]:
def predict_for_date(target_date, med):
    target_date = pd.to_datetime(target_date)
    last_date = df['datum'].max()

    if target_date <= last_date:
        row = df[df['datum'] == target_date]
        if len(row) > 0:
            return float(row[med].values[0])

    temp_df = df.copy()
    current_date = last_date + pd.Timedelta(days=1)

    while current_date <= target_date:
        new_row = {'datum': current_date}
        new_row['day_of_week']    = current_date.dayofweek
        new_row['month']          = current_date.month
        new_row['week_of_year']   = current_date.isocalendar()[1]
        new_row['quarter']        = current_date.quarter
        new_row['is_weekend']     = int(current_date.dayofweek >= 5)
        new_row['is_month_start'] = int(current_date.is_month_start)
        new_row['is_month_end']   = int(current_date.is_month_end)

        new_row['dow_sin']   = np.sin(2 * np.pi * new_row['day_of_week'] / 7)
        new_row['dow_cos']   = np.cos(2 * np.pi * new_row['day_of_week'] / 7)
        new_row['month_sin'] = np.sin(2 * np.pi * new_row['month'] / 12)
        new_row['month_cos'] = np.cos(2 * np.pi * new_row['month'] / 12)
        new_row['week_sin']  = np.sin(2 * np.pi * new_row['week_of_year'] / 52)
        new_row['week_cos']  = np.cos(2 * np.pi * new_row['week_of_year'] / 52)

        series = temp_df[med]
        new_row[f'{med}_lag1']   = series.iloc[-1]
        new_row[f'{med}_lag7']   = series.iloc[-7]  if len(series) >= 7  else series.mean()
        new_row[f'{med}_lag14']  = series.iloc[-14] if len(series) >= 14 else series.mean()
        new_row[f'{med}_lag30']  = series.iloc[-30] if len(series) >= 30 else series.mean()
        new_row[f'{med}_roll7']  = series.iloc[-7:].mean()
        new_row[f'{med}_roll14'] = series.iloc[-14:].mean()
        new_row[f'{med}_roll30'] = series.iloc[-30:].mean()
        new_row[f'{med}_std7']   = series.iloc[-7:].std()
        new_row[f'{med}_diff1']  = series.iloc[-1] - series.iloc[-2]
        new_row[f'{med}_diff7']  = series.iloc[-1] - series.iloc[-7] if len(series) >= 7 else 0

        med_features = results[med]['med_features']
        X_new = pd.DataFrame([new_row])[med_features]

        best_model = results[med]['xgb_model'] if results[med]['best'] == 'XGBoost' else results[med]['rf_model']

        # ✅ Clipper ici pour éviter la propagation de valeurs négatives dans les lags
        predicted_value = max(0.0, float(best_model.predict(X_new)[0]))

        new_row[med] = predicted_value
        temp_df = pd.concat([temp_df, pd.DataFrame([new_row])], ignore_index=True)

        current_date += pd.Timedelta(days=1)

    return round(predicted_value, 2)

In [20]:
def recommandation(med, target_date):
    predicted = predict_for_date(target_date, med)
    stock = current_stock[med]

    if stock < predicted:
        statut = "RUPTURE DE STOCK — RÉAPPROVISIONNER"
    else:
        statut = "STOCK SUFFISANT"

    return {"medicament": med, "statut": statut}

In [21]:
def predict_most_demanded(target_date):
    predictions = {}
    for med in meds:
        predictions[med] = predict_for_date(target_date, med)

    # Classer et retourner le plus demandé
    best = max(predictions, key=predictions.get)
    return {"medicament_plus_demande": best,
            "valeur_predite": round(predictions[best], 2)}

In [22]:
# Tester la prédiction globale
result = predict_most_demanded("2027-08-15")
print(result)

# Tester la recommandation
result = recommandation("N05C", "2027-08-15")
print(result)

{'medicament_plus_demande': 'N02BE', 'valeur_predite': 10.17}
{'medicament': 'N05C', 'statut': 'STOCK SUFFISANT'}


In [23]:
# Voir les métriques de chaque médicament
print(metrics_df)

  Médicament  RF MAE  RF MAPE(%)  RF R²  XGB MAE  XGB MAPE(%)  XGB R² Meilleur
0      M01AB    0.29        6.37  0.963     0.15         3.02   0.987  XGBoost
1      M01AE    0.23        5.94  0.943     0.12         3.06   0.981  XGBoost
2      N02BA    0.17        5.81  0.979     0.10         3.55   0.994  XGBoost
3      N02BE    2.09        7.20  0.949     1.22         4.11   0.983  XGBoost
4       N05B    0.63       10.66  0.946     0.30         4.04   0.989  XGBoost
5       N05C    0.05        5.80  0.983     0.02         2.32   0.997  XGBoost
6        R03    1.17       18.02  0.938     0.86        11.38   0.956  XGBoost
7        R06    0.41       12.25  0.927     0.19         5.12   0.977  XGBoost


In [24]:
import joblib

# Sauvegarder les modèles
for med in meds:
    joblib.dump(results[med]['rf_model'],  f'{med}_rf.pkl')
    joblib.dump(results[med]['xgb_model'], f'{med}_xgb.pkl')

# Sauvegarder les stocks et le dataframe
joblib.dump(current_stock, 'current_stock.pkl')
df.to_csv('df_engineered.csv', index=False)

print("✅ Tout est sauvegardé")

✅ Tout est sauvegardé


In [25]:
import zipfile, os

# Fichiers à inclure dans le zip
fichiers = (
    [f'{med}_rf.pkl'  for med in meds] +
    [f'{med}_xgb.pkl' for med in meds] +
    ['current_stock.pkl', 'df_engineered.csv']
)

with zipfile.ZipFile('models.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in fichiers:
        if os.path.exists(f):
            zf.write(f)
            print(f"✓ Ajouté : {f}")
        else:
            print(f"⚠️  Introuvable : {f}")

print(f"\n✅ models.zip créé ({os.path.getsize('models.zip') / 1e6:.1f} MB)")

✓ Ajouté : M01AB_rf.pkl
✓ Ajouté : M01AE_rf.pkl
✓ Ajouté : N02BA_rf.pkl
✓ Ajouté : N02BE_rf.pkl
✓ Ajouté : N05B_rf.pkl
✓ Ajouté : N05C_rf.pkl
✓ Ajouté : R03_rf.pkl
✓ Ajouté : R06_rf.pkl
✓ Ajouté : M01AB_xgb.pkl
✓ Ajouté : M01AE_xgb.pkl
✓ Ajouté : N02BA_xgb.pkl
✓ Ajouté : N02BE_xgb.pkl
✓ Ajouté : N05B_xgb.pkl
✓ Ajouté : N05C_xgb.pkl
✓ Ajouté : R03_xgb.pkl
✓ Ajouté : R06_xgb.pkl
✓ Ajouté : current_stock.pkl
✓ Ajouté : df_engineered.csv

✅ models.zip créé (10.0 MB)


In [26]:
from google.colab import files
files.download('models.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
# Installation des dépendances pour l'API
!pip install fastapi uvicorn pyngrok nest-asyncio pydantic --quiet

print("✅ Dépendances API installées")

✅ Dépendances API installées


In [33]:
# Imports et initialisation de l'application FastAPI
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import nest_asyncio
import uvicorn
from pyngrok import ngrok
from fastapi.middleware.cors import CORSMiddleware   # ← ajouter cet import

nest_asyncio.apply()  # nécessaire pour faire tourner uvicorn dans Colab

app = FastAPI(
    title="API Prédiction & Recommandation de Stock Pharmaceutique",
    description="Prédit la demande de médicaments et recommande un réapprovisionnement",
    version="1.0.0"
)
app.add_middleware(                  # ← ajouter ces lignes
    CORSMiddleware,                  # ←
    allow_origins=["*"],             # ←
    allow_methods=["*"],             # ←
    allow_headers=["*"],             # ←
)

MEDS_DISPONIBLES = meds  # réutilise la liste définie plus haut : meds = ['M01AB', ...]


# Schémas de requêtes/réponses (Pydantic) — définissent la forme des données retournées
class PredictionResponse(BaseModel):
    medicament: str
    date: str
    valeur_predite: float


class RecommandationResponse(BaseModel):
    medicament: str
    date: str
    stock_actuel: float
    valeur_predite: float
    statut: str


class MostDemandedResponse(BaseModel):
    date: str
    medicament_plus_demande: str
    valeur_predite: float
    toutes_les_predictions: dict


class StockResponse(BaseModel):
    stocks: dict


print("✅ Application FastAPI initialisée")

✅ Application FastAPI initialisée


In [29]:
# Fonctions utilitaires de validation
def _valider_medicament(med: str):
    if med not in MEDS_DISPONIBLES:
        raise HTTPException(
            status_code=404,
            detail=f"Médicament '{med}' inconnu. Médicaments disponibles : {MEDS_DISPONIBLES}"
        )


def _valider_date(date_str: str):
    try:
        pd.to_datetime(date_str)
    except Exception:
        raise HTTPException(
            status_code=400,
            detail=f"Date invalide : '{date_str}'. Format attendu : AAAA-MM-JJ (ex: 2027-08-15)"
        )


print("✅ Fonctions de validation prêtes")

✅ Fonctions de validation prêtes


In [34]:
# Endpoints de l'API

@app.get("/")
def accueil():
    return {
        "message": "Bienvenue sur l'API de prédiction et recommandation de stock",
        "endpoints": [
            "GET /predict?medicament=N05C&date=2027-08-15",
            "GET /recommandation?medicament=N05C&date=2027-08-15",
            "GET /predict-most-demanded?date=2027-08-15",
            "GET /stock"
        ],
        "medicaments_disponibles": MEDS_DISPONIBLES
    }


@app.get("/predict", response_model=PredictionResponse)
def predict(medicament: str, date: str):
    medicament = medicament.upper()
    _valider_medicament(medicament)
    _valider_date(date)

    valeur = predict_for_date(date, medicament)
    return PredictionResponse(
        medicament=medicament,
        date=date,
        valeur_predite=round(valeur, 2)
    )


@app.get("/recommandation", response_model=RecommandationResponse)
def recommandation_endpoint(medicament: str, date: str):
    medicament = medicament.upper()
    _valider_medicament(medicament)
    _valider_date(date)

    valeur = predict_for_date(date, medicament)
    stock = current_stock[medicament]

    if stock < valeur:
        statut = "RUPTURE DE STOCK — RÉAPPROVISIONNER"
    else:
        statut = "STOCK SUFFISANT"

    return RecommandationResponse(
        medicament=medicament,
        date=date,
        stock_actuel=float(stock),
        valeur_predite=round(valeur, 2),
        statut=statut
    )


@app.get("/predict-most-demanded", response_model=MostDemandedResponse)
def predict_most_demanded_endpoint(date: str):
    _valider_date(date)

    predictions = {med: round(predict_for_date(date, med), 2) for med in MEDS_DISPONIBLES}
    best = max(predictions, key=predictions.get)

    return MostDemandedResponse(
        date=date,
        medicament_plus_demande=best,
        valeur_predite=predictions[best],
        toutes_les_predictions=predictions
    )


@app.get("/stock", response_model=StockResponse)
def stock():
    return StockResponse(stocks={k: float(v) for k, v in current_stock.items()})


print("✅ Endpoints définis : /, /predict, /recommandation, /predict-most-demanded, /stock")

✅ Endpoints définis : /, /predict, /recommandation, /predict-most-demanded, /stock


In [35]:
# Lancement du serveur + tunnel ngrok
NGROK_AUTH_TOKEN = "3EAG0Vu3VdpLEFp4EENzN8jqoa4_4t5Gy6Ygj9ocnLys2Bkud"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

ngrok.kill()

public_url = ngrok.connect(8000)
print(f"🌍 API publique disponible ici : {public_url}")
print(f"📄 Documentation interactive (Swagger) : {public_url}/docs")

import threading

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("✅ Serveur FastAPI lancé en arrière-plan sur le port 8000")

🌍 API publique disponible ici : NgrokTunnel: "https://rescuer-gander-shrubbery.ngrok-free.dev" -> "http://localhost:8000"
📄 Documentation interactive (Swagger) : NgrokTunnel: "https://rescuer-gander-shrubbery.ngrok-free.dev" -> "http://localhost:8000"/docs
✅ Serveur FastAPI lancé en arrière-plan sur le port 8000


INFO:     Started server process [501]


In [36]:

#Test rapide de l'API depuis le notebook lui-même
import requests
import time

time.sleep(2)  # laisser le temps au serveur de démarrer

base_url = str(public_url).split('"')[1]  # extrait juste l'URL depuis l'objet NgrokTunnel
print("URL utilisée pour les tests :", base_url)

print("\n=== Test 1 : page d'accueil ===")
r = requests.get(f"{base_url}/")
print(r.status_code, r.json())

print("\n=== Test 2 : prédiction simple ===")
r = requests.get(f"{base_url}/predict", params={"medicament": "N05C", "date": "2027-08-15"})
print(r.status_code, r.json())

print("\n=== Test 3 : recommandation ===")
r = requests.get(f"{base_url}/recommandation", params={"medicament": "N05C", "date": "2027-08-15"})
print(r.status_code, r.json())

print("\n=== Test 4 : médicament le plus demandé ===")
r = requests.get(f"{base_url}/predict-most-demanded", params={"date": "2027-08-15"})
print(r.status_code, r.json())

print("\n=== Test 5 : stocks actuels ===")
r = requests.get(f"{base_url}/stock")
print(r.status_code, r.json())

print("\n=== Test 6 : médicament invalide (doit retourner 404) ===")
r = requests.get(f"{base_url}/predict", params={"medicament": "XXX", "date": "2027-08-15"})
print(r.status_code, r.json())

URL utilisée pour les tests : https://rescuer-gander-shrubbery.ngrok-free.dev

=== Test 1 : page d'accueil ===
INFO:     34.150.215.252:0 - "GET / HTTP/1.1" 200 OK
200 {'message': "Bienvenue sur l'API de prédiction et recommandation de stock", 'endpoints': ['GET /predict?medicament=N05C&date=2027-08-15', 'GET /recommandation?medicament=N05C&date=2027-08-15', 'GET /predict-most-demanded?date=2027-08-15', 'GET /stock'], 'medicaments_disponibles': ['M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06']}

=== Test 2 : prédiction simple ===
INFO:     105.190.139.159:0 - "GET / HTTP/1.1" 200 OK
INFO:     34.150.215.252:0 - "GET /predict?medicament=N05C&date=2027-08-15 HTTP/1.1" 200 OK
200 {'medicament': 'N05C', 'date': '2027-08-15', 'valeur_predite': 0.06}

=== Test 3 : recommandation ===
INFO:     105.190.139.159:0 - "GET / HTTP/1.1" 200 OK
INFO:     34.150.215.252:0 - "GET /recommandation?medicament=N05C&date=2027-08-15 HTTP/1.1" 200 OK
200 {'medicament': 'N05C', 'date': '2027-0